# SAUET Digital Wallet - Stress Test

End-to-end: cài đặt → clone repo GitHub → build SUT → setup MySQL → chạy test
(smoke/load/stress) → dump DB → reconcile → visualize.

**Yêu cầu**:
- Kaggle Settings: **Internet = On** (nếu `REPO_SOURCE=github`), Accelerator = None.
- Repo GitHub chứa 2 thư mục: `digitalwallet-system/`, `digitalwallet-stress-test/`.
- Session ≥ 1.5h cho chain đầy đủ.

**Thứ tự cell**: chạy tuần tự Cell 2 → 13. Mỗi cell idempotent hoặc tự kiểm tra
điều kiện (đã cài, đã clone, đã chạy).

## Clone 

sửa REPO_URL nếu cần

In [ ]:
import os, shutil, subprocess

# ==== Chọn nguồn repo ====
REPO_SOURCE = 'github'          # 'github' | 'kaggle'


REPO_URL    = 'https://github.com/vllddlvh/SAUETDigitalWallet.git'
REPO_BRANCH = ''                # '' = default


KAGGLE_DATASET_PATH = '/kaggle/input/datasets/nostagiguideus17/digital-wallet-source'
# =========================

REPO_DIR = '/tmp/SAUETDigitalWallet'


# 1. Download
if REPO_SOURCE == 'github':
    print(f'[repo] clone {REPO_URL} branch={REPO_BRANCH or "<default>"}')
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    cmd = ['git', 'clone', '--depth', '1']
    if REPO_BRANCH:
        cmd += ['--branch', REPO_BRANCH]
    cmd += [REPO_URL, REPO_DIR]
    subprocess.run(cmd, check=True)

elif REPO_SOURCE == 'kaggle':
    print(f'[repo] copy {KAGGLE_DATASET_PATH} → {REPO_DIR}')
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    shutil.copytree(KAGGLE_DATASET_PATH, REPO_DIR)

else:
    raise ValueError(f'REPO_SOURCE không hợp lệ: {REPO_SOURCE}')

SCRIPTS_DIR = f'{REPO_DIR}/digitalwallet-stress-test/scripts'
ORCH_DIR    = f'{REPO_DIR}/digitalwallet-stress-test'
open('/tmp/repo_dir',    'w').write(REPO_DIR)
open('/tmp/scripts_dir', 'w').write(SCRIPTS_DIR)
open('/tmp/orch_dir',    'w').write(ORCH_DIR)


# 2. Install dependencies
env = os.environ.copy()
env['SUT_DIR'] = REPO_DIR
print(f'\n$ bash {SCRIPTS_DIR}/00_install.sh')

## Setup

In [ ]:
def run_script(name, *args, env_extra=None):
    script = os.path.join(SCRIPTS_DIR, name)
    if not os.path.isfile(script):
        raise FileNotFoundError(script)
    env = os.environ.copy()
    env['SUT_DIR'] = REPO_DIR
    if env_extra:
        env.update(env_extra)
    cmd = ['bash', script, *args]
    print(f'\n$ {" ".join(cmd)}')
    r = subprocess.run(cmd, env=env, check=False)
    if r.returncode != 0:
        print(f'[{name}] exit={r.returncode}')
    return r.returncode

In [ ]:
import subprocess, os, sys

# Cài hệ thống.
rc = run_script('00_install.sh')
assert rc == 0, f'00_install.sh failed: {rc}'

# Application will be setup when test run (cell below). 
# Even through, run it here will save more time instead, in exchange, the DB will never be reset.

# rc = run_script('10_mysql.sh')
# assert rc == 0, f'10_mysql.sh failed: {rc}'

# rc = run_script('20_build_sut.sh')
# assert rc == 0, f'20_build_sut.sh failed: {rc}'

## Chain test

Cell 7 chạy **smoke → load → stress** tuần tự. Trước mỗi profile:
- `90_reset_db.sh` — kill SUT cũ + drop DB (được phép).
- `30_start_app.sh` — start SUT mới (kill instance dư nếu có).
- `src.run` — chỉ chạy k6 + sampler (không can thiệp SUT/DB).
- `80_dump_db.sh` — dump DB (all + violated).
- `reconcile.py` + `visualize.py` — phân tích.

Gate: pass khi exit k6 ∈ {0, 2} **và** `invariant_violation_count == 0`
**và** (chỉ smoke) `total_failures == 0`. Fail ⇒ dừng chain.

In [ ]:
import json, os, subprocess, sys
from datetime import datetime

SCRIPTS_DIR = open('/tmp/scripts_dir').read().strip()
ORCH_DIR    = open('/tmp/orch_dir').read().strip()
REPO_DIR    = open('/tmp/repo_dir').read().strip()

PROFILES  = ['smoke', 'load', 'stress']
BASE_OUT  = '/kaggle/working/results'
BASE_URL  = 'http://localhost:8080'
os.makedirs(BASE_OUT, exist_ok=True)

env_base = os.environ.copy()
env_base['SUT_DIR']     = REPO_DIR
env_base['BASE_URL']    = BASE_URL
env_base['RESULTS_DIR'] = BASE_OUT


# ── Helpers ─────────────────────────────────────────────────────────────────

def run_script(name, *args):
    script = os.path.join(SCRIPTS_DIR, name)
    if not os.path.isfile(script):
        print(f'[skip] {script} không tồn tại')
        return -1
    cmd = ['bash', script, *args]
    print(f'\n$ {" ".join(cmd)}')
    return subprocess.run(cmd, env=env_base, check=False).returncode


def run_python(module_or_script, *args):
    if module_or_script.endswith('.py'):
        cmd = [sys.executable, module_or_script, *args]
    else:
        cmd = [sys.executable, '-m', module_or_script, *args]
    print(f'\n$ {" ".join(cmd)}')
    return subprocess.run(cmd, env=env_base, cwd=ORCH_DIR, check=False).returncode


def gate(profile, rc, summary):
    if rc not in (0, 2):
        return False, f'orchestrator exit={rc}'
    viol = summary.get('violation', 0)
    fail = summary.get('failure', 0)
    if viol > 0:
        return False, f'{viol} violation'
    if profile == 'smoke' and fail > 0:
        return False, f'{fail} failure'
    return True, 'ok'


# ── Chain runner ────────────────────────────────────────────────────────────

results = []
broken_at = None

for profile in PROFILES:
    ts      = datetime.now().strftime('%Y%m%dT%H%M%S')
    out_dir = f'{BASE_OUT}/{profile}_{ts}'
    os.makedirs(out_dir, exist_ok=True)
    print(f'\n===== {profile.upper()}  out={out_dir} =====')

    # 1. Reset DB
    rc = run_script('90_reset_db.sh')
    if rc != 0:
        results.append({'profile': profile, 'out_dir': out_dir, 'exit': rc,
                        'pass': False, 'reason': f'90_reset_db exit={rc}', 'summary': {}})
        broken_at = profile; break

    # 2. Start SUT
    rc = run_script('30_start_app.sh')
    if rc != 0:
        results.append({'profile': profile, 'out_dir': out_dir, 'exit': rc,
                        'pass': False, 'reason': f'30_start_app exit={rc}', 'summary': {}})
        broken_at = profile; break

    # 3. k6 (chỉ k6 + sampler)
    rc = run_python('src.run', '--profile', profile, '--out', out_dir,
                    '--base-url', BASE_URL)

    k6_csv = f'{out_dir}/k6_raw.csv'
    k6_ok  = os.path.exists(k6_csv)

    summary = {}
    if k6_ok:
        # 4. Dump DB (chỉ deadAccount)
        run_script('80_dump_db.sh', '--mode=violated', f'--from-csv={k6_csv}',
                   f'--out={out_dir}', '--grace=0')

        # 5. Reconcile → request_log.csv + summary.json
        run_python(f'{ORCH_DIR}/src/reconcile.py',
                   '--k6-csv', k6_csv, '--out-dir', out_dir)

        # 6. Visualize
        run_python(f'{ORCH_DIR}/src/visualize.py',
                   '--log-csv', f'{out_dir}/request_log.csv',
                   '--sampler-csv', f'{out_dir}/sampler.csv',
                   '--out-dir', out_dir)

        sp = f'{out_dir}/summary.json'
        if os.path.exists(sp):
            with open(sp) as f:
                summary = json.load(f)
    else:
        print(f'[skip] {k6_csv} không tồn tại')

    # 7. Gate + append kết quả
    if not k6_ok:
        ok, reason = False, f'k6_raw.csv không được sinh (orchestrator exit={rc})'
    else:
        ok, reason = gate(profile, rc, summary)
    print(f'\n[gate] {profile}: {"PASS" if ok else "FAIL"} — {reason}')

    results.append({
        'profile': profile,
        'out_dir': out_dir,
        'exit':    rc,
        'pass':    ok,
        'reason':  reason,
        'summary': {k: summary.get(k) for k in
                    ('total_requests', 'success', 'failure', 'violation',
                     'success_rate', 'failure_rate', 'violation_rate')},
    })

    if not ok:
        broken_at = profile
        print(f'[STOP] {profile} FAIL — dừng chain.')
        break

# Lưu & in tổng hợp
with open('/tmp/chain_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('\n======== Tổng hợp ============================')
if not results:
    print('  (không có profile nào được chạy)')
for r in results:
    s = r['summary']
    print(f"  [{'PASS' if r['pass'] else 'FAIL'}] {r['profile']:7s}  "
          f"exit={r['exit']}  "
          f"req={s.get('total_requests', '?')}  "
          f"ok={s.get('success', '?')}  "
          f"fail={s.get('failure', '?')}  "
          f"viol={s.get('violation', '?')}  "
          f"— {r['reason']}")
if broken_at:
    print(f'\n⚠ Chain dừng ở "{broken_at}".')

## Kết quả

Hiển thị summary Markdown + toàn bộ plot. Nếu có `invariant_violation`,
bảng lỗi ở cuối `reconcile_summary.md` liệt kê `wallet_id` để đối chiếu
với `violated_accounts.csv` / `violated_transactions.csv`.

In [ ]:
from IPython.display import display, Markdown, Image
import glob, json, os

with open('/tmp/chain_results.json') as f:
    results = json.load(f)

for r in results:
    p, d = r['profile'], r['out_dir']
    display(Markdown(f"# PROFILE: {p.upper()} — {'PASS' if r['pass'] else 'FAIL'} ({r['reason']})"))
    
    if not os.path.exists(f'{d}/plots'):
        display(Markdown("*(Không có thư mục plots)*"))
        display(Markdown('---'))
        continue

    # 1. BIỂU ĐỒ KẾT QUẢ TEST (Success / Failure / Violation)
    display(Markdown("### 1. Kết quả Test (Success / Failure / Violation)"))
    test_plots = [
        'count_success.png', 'count_failure.png', 'count_violation.png',
        'ratio_success.png', 'ratio_failure.png', 'ratio_violation.png',
        'combined_count.png', 'pie_overall.png'
    ]
    for png_name in test_plots:
        png_path = os.path.join(d, 'plots', png_name)
        if os.path.exists(png_path):
            display(Markdown(f"**{png_name}**"))
            display(Image(filename=png_path))
        else:
            print(f'Missing: {png_name}')

    # 2. BIỂU ĐỒ TÀI NGUYÊN HỆ THỐNG (CPU / RAM / JVM / MySQL)
    display(Markdown("### 2. Tài nguyên Hệ thống (Monitor)"))
    sys_plots = [
        'system_cpu.png', 'system_ram.png',
        'system_jvm_memory.png', 'system_mysql_threads.png'
    ]
    for png_name in sys_plots:
        png_path = os.path.join(d, 'plots', png_name)
        if os.path.exists(png_path):
            display(Markdown(f"**{png_name}**"))
            display(Image(filename=png_path))
        else:
            print(f'Missing: {png_name}')

    display(Markdown('---'))

## Ending

- Copy kết quả về `/kaggle/working/results_final/` để commit.
- Kill app + mysqld (Kaggle commit mode dừng mọi background process khi cell cuối
  kết thúc, nhưng kill chủ động để tránh rò rỉ trong notebook session).

Chỉ chạy cell này khi đã xem xong kết quả. Nếu chạy tiếp một test khác, **không**
chạy Cell 13 — quay lại Cell 7 để đổi `PROFILE` và chạy Cell 8.

In [ ]:
import json, os, shutil, subprocess

SCRIPTS_DIR = open('/tmp/scripts_dir').read().strip()

with open('/tmp/chain_results.json') as f:
    results = json.load(f)

dest_root = '/kaggle/working/results_final'
os.makedirs(dest_root, exist_ok=True)
for r in results:
    src, dest = r['out_dir'], f'{dest_root}/{r["profile"]}'
    if os.path.abspath(src) != os.path.abspath(dest):
        shutil.copytree(src, dest, dirs_exist_ok=True)
        print(f'copied: {src} → {dest}')

with open(f'{dest_root}/chain_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f'\nArchived {len(results)} profile under {dest_root}.')
for r in results:
    d = f'{dest_root}/{r["profile"]}'
    n_files = len([x for x in os.listdir(d) if os.path.isfile(os.path.join(d, x))])
    n_png   = len(os.listdir(f'{d}/plots')) if os.path.isdir(f'{d}/plots') else 0
    print(f'  {r["profile"]:7s}  {"PASS" if r["pass"] else "FAIL"}  '
          f'{n_files} files + {n_png} plots')

# Cleanup chủ đích: kill SUT + drop DB.
print('\n$ bash scripts/90_reset_db.sh')
subprocess.run(['bash', os.path.join(SCRIPTS_DIR, '90_reset_db.sh')], check=False)
print('cleanup done. mysqld sẽ chết khi Kaggle session kết thúc.')